# Lab 2 — Comparing LLM Responses

This lab builds on my first API experiments by comparing how different OpenAI models respond to the same question.

**Focus:** model comparison, reusable functions, response collection, and LLM-based evaluation.

*Based on concepts from Ed Donner's Agentic AI course, with my own modifications.*


## 1. Setup

I’m using Python environment variables for API access and the OpenAI client for the model calls.


In [2]:
# os + dotenv: read the API key from the environment, not from this notebook.
# OpenAI: make chat completions. json: parse the judge model's ranking later.
# Markdown/display: show model replies in a readable way in Jupyter.

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display


In [3]:
# Load .env into the process. override=True so this notebook's key wins if one was already set.
load_dotenv(override=True)


True

In [ ]:
# This is just to chcek for other keys in my environment. But since i am using only OpenAI. iT WAS JUST A TEST.
# Confirm keys loaded without printing the secret. This notebook only calls OpenAI.
# the other checks are leftover from the course lab and are optional here.

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print("OpenAI API Key is available!!!!! Yeeee")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print("Anthropic API Key is available.")
else:
    print("Anthropic API Key not set (So sad. Need to make more money)")

if google_api_key:
    print("Google API Key is available.")
else:
    print("Google API Key not set (so sad. Need to make more mone)")

if deepseek_api_key:
    print("DeepSeek API Key is available.")
else:
    print("DeepSeek API Key not set (so sad. Need to make more mone)")

if groq_api_key:
    print("Groq API Key is available.")
else:
    print("Groq API Key not set (so sad. Need to make more mone)")

if grok_api_key:
    print("Grok API Key is available.")
else:
    print("Grok API Key not set (Optional But so sad. Need to make more mone)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}") 
else:
    print("OpenRouter API Key not set (Optional. So sad. Need to make more mone)")


OpenAI API Key is available!!!!! Yeeee
Anthropic API Key not set (So sad. Need to make more money)
Google API Key not set (so sad.  Need to make more mone)
DeepSeek API Key not set (so sad.  Need to make more mone)
Groq API Key not set (so sad.  Need to make more mone)
Grok API Key not set (Optional But so sad.  Need to make more mone)
OpenRouter API Key not set (Optional. so sad.  Need to make more mone)


## 2. Generate a question

Instead of comparing models on different prompts, I use one challenging question so the responses can be compared on the same task.


In [5]:
# Ask a model to invent ONE short, thought-provoking question.
# I generate the test question instead of writing it myself, then reuse it for every comparison.

request = """
Please come up with a challenging, nuanced question with a succinct answer
that I can ask several versions of ChatGPT, so I can compare how they think.
Not a mathematical puzzle, but a thought-provoking question that needs intelligent insight.
Include in your question that the answer must be short.
"""


request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]


In [6]:
# Sanity-check: chat APIs take a list of {role, content} dicts, not a bare string.
messages


[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer\nthat I can ask several versions of ChatGPT, so I can compare how they think.\nNot a mathematical puzzle, but a thought-provoking question that needs intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [7]:
# Create the OpenAI client (reads OPENAI_API_KEY).
# gpt-5.6-sol writes the shared test question; later cells only change the model ID.

openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5.6-sol", messages=messages)
question = response.choices[0].message.content

display(Markdown(question))


An institution makes the right decision for reasons it cannot disclose: silence will undermine public trust, but disclosure will cause preventable harm. What does it owe the public—truth, justification, or restraint—and why? Answer in no more than three sentences.

## 3. Compare different OpenAI models

I kept the provider constant and changed only the model ID. This makes the comparison focused on how different model versions approach the same question.


In [8]:
# ChatGPT versions I will compare, cheapest to stronger.
# API IDs are lowercase with hyphens (not display names like "GPT-5.4 Mini").

model_name = "gpt-5.4-nano"
model_name0 = "gpt-5.4-mini"
model_name1 = "gpt-5.5"
model_name2 = "gpt-5.6-luna"
model_name3 = "gpt-5.6-terra"


In [9]:
# competitors/answers: parallel lists so I can zip model ID with its reply for judging.
# messages now holds the generated question, the same prompt for every model.

competitors = []
answers = []
messages = [{"role": "user", "content": question}]


In [10]:
# One helper so every comparison cell stores results the same way (needed by the judge later).

def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))


In [11]:
# Smallest/cheapest model in the set. Same messages; only the model ID changes.
# reasoning_effort="none" keeps this comparison closer to a direct answer.

model_name = "gpt-5.4-nano"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)


It owes **restraint**, because when disclosure would predictably cause preventable harm and no disclosable justification is possible, truth cannot be served by speaking. Even if silence may feel like an abandonment of transparency, the institution should aim to protect the public first and minimize wrongdoing rather than reveal information it cannot safely share. So the obligation is not simply truth or justification, but **a harm-aware restraint that preserves public safety and maintains trust through whatever limited, non-damaging information can be responsibly provided**.

It owes **restraint**, not full truth or full justification in disclosable form, because the duty is to avoid foreseeable, preventable harm when silence would erode trust. The appropriate approach is to be as transparent as possible **without** disclosing sensitive details, making clear that reasons are withheld for safety or other non-negotiable constraints, thereby preserving trust while minimizing harm.

In [12]:
# Next size up: still cheap, usually a bit stronger than nano. Same question.

model_name = "gpt-5.4-mini"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)


It owes the public restraint in disclosure, but not silence without accountability. When telling the truth would foreseeably cause preventable harm, the institution should withhold the details while still offering as much justified explanation as it can—enough to show the decision is principled, limited, and reviewable. What it owes, then, is a balance of honesty and care: minimal disclosure, maximally accountable reasoning, and a clear commitment to be answerable later if the reasons can safely be revealed.

In [13]:
# Mid-tier GPT-5.5, still OpenAI-only so provider is not a confounder.

model_name = "gpt-5.5"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)


It owes the public restraint plus as much truthful justification as can be given without causing the harm. Democratic trust does not require revealing every operative fact, but it does require honesty about the limits of disclosure, the standards used, who is accountable, and when independent review or later disclosure will occur. Silence should not be a blank check; it must be paired with mechanisms that let the public know the secrecy is constrained, temporary where possible, and answerable.

In [14]:
# GPT-5.6 Luna: cost-focused 5.6 variant, still with reasoning_effort none.

model_name = "gpt-5.6-luna"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)


It owes the public a truthful account of what it can safely disclose, not necessarily every underlying reason. It should explain the decision’s governing principles, evidence, and safeguards without revealing details whose disclosure would cause preventable harm. This calibrated transparency preserves accountability and trust while honoring its duty to minimize harm.

In [15]:
# GPT-5.6 Terra: stronger 5.6 tier. low reasoning to see if extra thinking changes the answer.

model_name = "gpt-5.6-terra"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="low")
answer = response.choices[0].message.content

record(model_name, answer)


It owes the public as much truthful justification as can be given without creating the preventable harm, plus a clear acknowledgment of the limits on disclosure. Trust cannot rest on demands for transparency at any cost, but neither can secrecy be self-validating; the institution must accept oversight by parties able to assess its reasons confidentially. Restraint is justified only when it protects the public rather than the institution’s convenience or power.

## 4. Organize the responses

Once the responses are collected, I organize them so they can be reviewed and evaluated as a group.


In [16]:
# Confirm I collected one reply per model before judging (expect 5).

print(len(competitors))
print(competitors)
print(answers)


6
['gpt-5.4-nano', 'gpt-5.4-nano', 'gpt-5.4-mini', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-terra']
['It owes **restraint**, because when disclosure would predictably cause preventable harm and no disclosable justification is possible, truth cannot be served by speaking. Even if silence may feel like an abandonment of transparency, the institution should aim to protect the public first and minimize wrongdoing rather than reveal information it cannot safely share. So the obligation is not simply truth or justification, but **a harm-aware restraint that preserves public safety and maintains trust through whatever limited, non-damaging information can be responsibly provided**.', 'It owes **restraint**, not full truth or full justification in disclosable form, because the duty is to avoid foreseeable, preventable harm when silence would erode trust. The appropriate approach is to be as transparent as possible **without** disclosing sensitive details, making clear that reasons are withheld for 

In [17]:
# zip keeps each model ID next to its own answer (lists stay in call order).

for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5.4-nano

It owes **restraint**, because when disclosure would predictably cause preventable harm and no disclosable justification is possible, truth cannot be served by speaking. Even if silence may feel like an abandonment of transparency, the institution should aim to protect the public first and minimize wrongdoing rather than reveal information it cannot safely share. So the obligation is not simply truth or justification, but **a harm-aware restraint that preserves public safety and maintains trust through whatever limited, non-damaging information can be responsibly provided**.
Competitor: gpt-5.4-nano

It owes **restraint**, not full truth or full justification in disclosable form, because the duty is to avoid foreseeable, preventable harm when silence would erode trust. The appropriate approach is to be as transparent as possible **without** disclosing sensitive details, making clear that reasons are withheld for safety or other non-negotiable constraints, ther

In [18]:
# Build one combined text block containing all model responses.


together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"


In [19]:
# Display the combined responses before evaluation.

print(together)


# Response from competitor 1

It owes **restraint**, because when disclosure would predictably cause preventable harm and no disclosable justification is possible, truth cannot be served by speaking. Even if silence may feel like an abandonment of transparency, the institution should aim to protect the public first and minimize wrongdoing rather than reveal information it cannot safely share. So the obligation is not simply truth or justification, but **a harm-aware restraint that preserves public safety and maintains trust through whatever limited, non-damaging information can be responsibly provided**.

# Response from competitor 2

It owes **restraint**, not full truth or full justification in disclosable form, because the duty is to avoid foreseeable, preventable harm when silence would erode trust. The appropriate approach is to be as transparent as possible **without** disclosing sensitive details, making clear that reasons are withheld for safety or other non-negotiable constrai

## 5. Use an LLM as a judge

The next step is to evaluate the responses rather than relying only on manual inspection.

I create a judging prompt that asks another model to rank the responses against the same criteria.


In [20]:
# Build a structured prompt for evaluating the model responses.

judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [21]:
# Inspect the judging prompt before sending it to the evaluator.

print(judge)


You are judging a competition between 6 competitors.
Each model has been given this question:

An institution makes the right decision for reasons it cannot disclose: silence will undermine public trust, but disclosure will cause preventable harm. What does it owe the public—truth, justification, or restraint—and why? Answer in no more than three sentences.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

It owes **restraint**, because when disclosure would predictably cause preventable harm and no disclosable justification is possible, truth cannot be served by speaking. Even if silence may feel like an abandonment of transparency, the institution should aim to protect the 

In [22]:
# Prepare the judging prompt as an API message.

judge_messages = [{"role": "user", "content": judge}]


## 6. Ranking the results

I use another model to evaluate the responses and return a structured ranking.


# Use a separate model to evaluate and rank the responses.
## And now for 6-astra!

Branded as "The most truth-seeking large language model in the world".. NOT REALLY BUT GIVEN ITS THE LATEST VERSION WE CAN SAY THAT

In [ ]:
# Parse the evaluator's JSON result and display the final ranking.

# Judgement time!
# We are using gpt-6-astra is "The most truth-seeking large language model in the world." NOT REALLY BUT GIVEN ITS THE LATEST VERSION WE CAN SAY THAT



model_name = "gpt-6-astra"

response = openai.chat.completions.create(model=model_name, messages=judge_messages)
results = response.choices[0].message.content
print(results)


{"results":["6","4","3","5","2","1"]}


In [24]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gpt-5.6-terra
Rank 2: gpt-5.5
Rank 3: gpt-5.4-mini
Rank 4: gpt-5.6-luna
Rank 5: gpt-5.4-nano
Rank 6: gpt-5.4-nano


## Key Takeaway

This lab helped me move from making individual LLM calls to designing a small **multi-model evaluation workflow**.

The pattern is:

**Generate → Compare → Evaluate → Rank**

This is an early example of how multiple LLM calls can work together as part of an agentic system.
